In [1]:
import pybamm
import numpy as np
import pandas as pd
from plotly.subplots import make_subplots
import plotly.graph_objects as go

v_nom = 3.63 # V

In [2]:
experiment = pybamm.Experiment(
    [
        (
            "Discharge at 0.2C until 2.5V",
            "Hold at 2.5V until 0.05A"
        )
    ],
    period="1 second"
)
model = pybamm.lithium_ion.SPMe()
sim = pybamm.Simulation(model, experiment=experiment, parameter_values=pybamm.ParameterValues("Chen2020"))
sol = sim.solve(initial_soc=1)
discharge_capacity = np.sum(sol["Power [W]"].data) / 3600
print(discharge_capacity)

18.867593577409025


In [3]:
c_rates_charge = np.arange(0.1, 0.8, 0.1) 
c_rates_discharge = np.array([0.1, 0.2, 0.3, 0.5, 0.7, 1.0, 1.5]) 
n_c = []
n_c_at_vnom = []
solutions_charge = []
for i in c_rates_charge:
    experiment = pybamm.Experiment(
        [
            (
                f"Charge at {i}C until 4.2V"
            )
        ]
    )
    model = pybamm.lithium_ion.SPMe()
    sim = pybamm.Simulation(model, experiment=experiment, parameter_values=pybamm.ParameterValues("Chen2020"))
    sol = sim.solve(initial_soc=0)
    solutions_charge.append(sol)
    new_n_c = 1 - sol["Current [A]"].data * (sol["Local ECM resistance [Ohm]"].data) / sol["Terminal voltage [V]"].data
    n_c.append(new_n_c)
    n_c_at_vnom.append(new_n_c[np.argmin(np.abs(sol["Terminal voltage [V]"].data - v_nom))])
n_d = []
n_d_at_vnom = []
solutions_discharge = []
for i in c_rates_discharge:
    experiment = pybamm.Experiment(
        [
            (
                f"Discharge at {i}C until 2.5V",
            )
        ]
    )
    model = pybamm.lithium_ion.SPMe()
    sim = pybamm.Simulation(model, experiment=experiment, parameter_values=pybamm.ParameterValues("Chen2020"))
    sol = sim.solve(initial_soc=1)
    solutions_discharge.append(sol)
    new_n_d = 1 - sol["Current [A]"].data * (-sol["Local ECM resistance [Ohm]"].data) / sol["Terminal voltage [V]"].data
    n_d.append(new_n_d)
    n_d_at_vnom.append(new_n_d[np.argmin(np.abs(sol["Terminal voltage [V]"].data - v_nom))])
charging_efficiency = n_c_at_vnom[2]
discharging_efficiency = n_d_at_vnom[1]
print(charging_efficiency)
print(discharging_efficiency)

0.977878085872755
1.014364528784435


In [13]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.03)
fig.update_layout(
    width=800,
    height=600,
    margin={"r": 0, "l": 0, "t": 0.9, "b": 0},
    paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
    font_size=14,
    legend2={"y": 0.5, "yanchor": "top"},
)
fig.update_xaxes(
    {"showline": True, "linewidth": 1, "linecolor": "black", "ticks":"outside"}
)
fig.update_yaxes(
    {"showline": True, "linewidth": 1, "linecolor": "black", "ticks":"outside"}
)
for i in range(len(n_c)):
    fig.add_trace(
        go.Scatter(x=solutions_charge[i]["Terminal voltage [V]"].data[1:], y=n_c[i][1:], name=f"{c_rates_charge[i]:.1f}C", legend="legend", mode="lines"), row=1, col=1 
    )
fig.add_vline(x=v_nom, line_dash="dash")
fig.add_trace(go.Scatter(x=[v_nom], y=[charging_efficiency], marker_symbol="x", marker_color = "red", marker_size=15, showlegend=False), row=1, col=1)
fig.update_yaxes(row=1, col=1, title_text="$\\text{Charging efficiency } \\eta_c$")
for i in range(len(n_d)):
    fig.add_trace(
        go.Scatter(x=solutions_discharge[i]["Terminal voltage [V]"].data[1:], y=n_d[i][1:], name=f"-{c_rates_discharge[i]:.1f}C", legend="legend2", mode="lines"), row=2, col=1 
    )
fig.add_vline(x=v_nom, line_dash="dash")
fig.add_trace(go.Scatter(x=[v_nom], y=[discharging_efficiency], marker_symbol="x", marker_color = "red", marker_size=15, showlegend=False), row=2, col=1)
fig.update_xaxes(row=2, col=1, title="$\\text{Terminal Voltage } V$", ticksuffix="V")
fig.update_yaxes(row=2, col=1, title_text="$\\text{Discharging efficiency } \\eta_d$")
fig.write_image("../thesis/Images/charging_efficiency.pdf", "pdf")
fig.show()

In [5]:
charge_currents = np.arange(0.3, 3.7, 0.2)
charge_energies = []
for current in charge_currents:
    experiment = pybamm.Experiment(
        [
            (
                f"Charge at {current}A until 4.2V",
                "Rest for 5 minutes",
                "Discharge at 0.2C until 2.5V",
                "Hold at 2.5V until 0.05A",
            )
        ],
        period="1 second"
    )
    model = pybamm.lithium_ion.SPMe()
    sim = pybamm.Simulation(model=model, experiment=experiment, parameter_values=pybamm.ParameterValues("Chen2020"))
    sol = sim.solve(initial_soc=0)
    powers = sol["Power [W]"].data
    powers = powers[powers > 0]
    charge_energies.append(np.sum(powers) / 3600 * discharging_efficiency)
charge_energies = np.array(charge_energies)

In [6]:
discharge_currents = np.arange(0.3, 7.7, 0.4)
discharge_energies = []
for current in discharge_currents:
    experiment = pybamm.Experiment(
        [
            (
                f"Discharge at {current}A until 2.5V",
                "Rest for 5 minutes",
                "Discharge at 0.2C until 2.5V",
                "Hold at 2.5V until 0.05A",
            )
        ],
        period="1 second"
    )
    model = pybamm.lithium_ion.SPMe()
    sim = pybamm.Simulation(model=model, experiment=experiment, parameter_values=pybamm.ParameterValues("Chen2020"))
    sol = sim.solve(initial_soc=1)
    powers = sol["Power [W]"].data
    last_zero_index = np.where(powers == 0.0)[0][-1]
    powers = powers[last_zero_index + 1:]
    discharge_energies.append(np.sum(powers) / 3600 * discharging_efficiency)
discharge_energies = np.array(discharge_energies)

In [7]:
v_upper = 19.14
u_upper = np.sum(charge_currents * (charge_energies - v_upper)) / np.sum(charge_currents**2)
print(u_upper, v_upper)
v_lower = 0.0
u_lower = np.sum(discharge_currents * (discharge_energies - v_lower)) / np.sum(discharge_currents**2)
print(u_lower, v_lower)

-1.325616130163523 19.14
0.08746292503362288 0.0


In [20]:
fig = make_subplots(rows=2, cols=1, vertical_spacing=0.1)
fig.update_layout(
    width=800,
    height=700,
    margin={"r": 0, "l": 0, "t": 0.9, "b": 0},
    paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
    font_size=14,
)
fig.update_xaxes(
    showline=True, linewidth=1, linecolor="black", ticks="outside", title="$\\text{Charge Current } I$", ticksuffix="A", range=[0, 3.7], row=1, col=1
)
fig.update_yaxes(
    showline=True, linewidth=1, linecolor="black", ticks="outside", title="$\\text{Upper Energy Limit } a_2$", ticksuffix="Wh",  row=1, col=1
)
fig.add_trace(
    go.Scatter(x=charge_currents, y=charge_energies, line_color="green", showlegend=False, mode="lines"), row=1, col=1
)
fig.add_trace(
    go.Scatter(x=np.arange(0, 4, 0.1), y=(u_upper*np.arange(0, 4, 0.1) + v_upper), showlegend=False, mode="lines", line_color="green", line_dash="dash"), row=1, col=1
)
fig.update_xaxes(
    showline=True, linewidth=1, linecolor="black", ticks="outside", title="$\\text{Discharge Current } I$", ticksuffix="A", range=[0, 7.9], row=2, col=1
)
fig.update_yaxes(
    showline=True, linewidth=1, linecolor="black", ticks="outside", title="$\\text{Lower Energy Limit } a_1$", ticksuffix="Wh", row=2, col=1
)
fig.add_trace(
    go.Scatter(x=discharge_currents, y=discharge_energies, line_color="red", showlegend=False, mode="lines"), row=2, col=1
)
fig.add_trace(
    go.Scatter(x=np.arange(0.0, 10, 0.1), y=(u_lower*np.arange(0.0, 10, 0.1) + v_lower), showlegend=False, mode="lines", line_color="red", line_dash="dash"), row=2, col=1
)
fig.write_image("../thesis/Images/energy_limits.pdf", "pdf")
fig.show()

In [16]:
df = pd.read_csv("data/SoC_Values.csv", index_col=0)
ocv_values = df["ocv_values"].to_numpy()
soc_values = df["soc_values"].to_numpy()
fig = go.Figure()
fig.update_layout(
    width=800,
    height=300,
    margin={"r": 0, "l": 0, "t": 0.9, "b": 0},
    paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
    font_size=14,
)
fig.update_xaxes(
    showline=True, linewidth=1, linecolor="black", ticks="outside", title="$\\text{State-of-Charge}$", ticksuffix="%"
)
fig.update_yaxes(
    showline=True, linewidth=1, linecolor="black", ticks="outside", title="$\\text{Open-Circuit Voltage}$", ticksuffix="V"
)
fig.add_trace(
    go.Scatter(x=soc_values * 100, y=ocv_values)
)
fig.write_image("../thesis/Images/soc_estimation.pdf", "pdf")
fig.show()

In [14]:
experiment = pybamm.Experiment(
    [
        (
            "Charge at 0.7C until 4.2V",
            "Hold at 4.2V until 0.05A"
        )
    ],
    period = "1 second",
)
charging_sim = pybamm.Simulation(
    experiment=experiment,
    model=pybamm.lithium_ion.SPMe(),
    parameter_values=pybamm.ParameterValues("Chen2020"),
)
sol = charging_sim.solve(initial_soc=0)
ocv_charging = sol["Battery open-circuit voltage [V]"].data
max_power_charging = - sol["Power [W]"].data

experiment = pybamm.Experiment(
    [
        (
            "Discharge at 1.5C until 2.5V",
            "Hold at 2.5V until 0.05A"
        )
    ],
    period = "1 second",
)
discharging_sim = pybamm.Simulation(
    experiment=experiment,
    model=pybamm.lithium_ion.SPMe(),
    parameter_values=pybamm.ParameterValues("Chen2020"),
)
sol = discharging_sim.solve(initial_soc=1)
ocv_discharging = sol["Battery open-circuit voltage [V]"].data[::-1]
max_power_discharging = sol["Power [W]"].data[::-1]

In [18]:
fig = go.Figure()
fig.update_layout(
    width=800,
    height=400,
    margin={"r": 0, "l": 0, "t": 0.9, "b": 0},
    paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
    font_size=14,
)
fig.update_xaxes(
    showline=True, linewidth=1, linecolor="black", ticks="outside", title="$\\text{Open-Circuit Voltage}$", ticksuffix="V"
)
fig.update_yaxes(
    showline=True, linewidth=1, linecolor="black", ticks="outside", title="$\\text{Power Limits}$", ticksuffix="W"
)
fig.add_trace(
    go.Scatter(x=ocv_charging, y=max_power_charging, line_color="green", name="Charging", mode="lines")
)
fig.add_trace(
    go.Scatter(x=ocv_discharging, y=max_power_discharging, line_color="red", name="Discharging", mode="lines")
)
fig.write_image("../thesis/Images/power_limits.pdf", "pdf")
fig.show()